In [1]:
!uv pip install sentence-transformers rank-bm25 openai anthropic \
            tenacity omegaconf rich jsonlines tqdm

Using Python 3.12.13 environment at: /usr
Checked 9 packages in 205ms


In [2]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    !pip install unsloth vllm
else:
    pass # For Colab / Kaggle, we need extra instructions hidden below \/

In [3]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
!pip install --upgrade -qqq uv
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install!
    !pip install unsloth vllm
else:
    !uv pip uninstall transformers trl
    try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
    except: _numpy = "numpy"; _pil = "pillow"
    try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except: is_t4 = False
    _vllm, _triton = ('vllm==0.9.2', 'triton==3.2.0') if is_t4 else ('vllm==0.15.1', 'triton')
    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

In [4]:
!uv pip install --upgrade torch
!uv pip install "vllm<=0.10.0" "transformers<4.54.0"

Using Python 3.12.13 environment at: /usr
Resolved 190 packages in 1.42s
Prepared 85 packages in 1m 49s
Uninstalled 37 packages in 947ms
Installed 85 packages in 1.04s
 + apache-tvm-ffi==0.1.9
 + cbor2==6.1.2
 - compressed-tensors==0.10.2
 + compressed-tensors==0.17.0
 - cryptography==48.0.1
 + cryptography==49.0.0
 - cuda-bindings==12.9.7
 + cuda-bindings==13.3.1
 - cuda-core==0.3.2
 + cuda-core==1.0.1
 - cuda-python==12.9.7
 + cuda-python==13.3.1
 + cuda-tile==1.3.0
 - cuda-toolkit==12.8.1
 + cuda-toolkit==13.0.2
 - depyf==0.18.0
 + depyf==0.20.0
 - dill==0.4.0
 + dill==0.4.1
 - fastapi==0.137.2
 + fastapi==0.136.3
 + fastsafetensors==0.3.2
 + flashinfer-cubin==0.6.12
 + flashinfer-python==0.6.12
 - fsspec==2025.9.0
 + fsspec==2026.6.0
 - grpcio==1.81.0
 + grpcio==1.81.1
 - huggingface-hub==0.36.2
 + huggingface-hub==1.19.0
 + humming-kernels==0.1.4
 + ijson==3.5.0
 + jmespath==1.1.0
 - llguidance==0.7.30
 + llguidance==1.7.6
 - llvmlite==0.44.0
 + llvmlite==0.47.0
 - lm-format-enfor

In [5]:
import os
import sys
from IPython import get_ipython


def configure_environment_paths():
    try:
        if "google.colab" in str(get_ipython()):
            print("✅ Environment: Google Colab")
            base_data_path = "/content/"
            base_output_path = "/content/"
            environment_name = "colab"
        elif os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            print("✅ Environment: Kaggle")
            base_data_path = "/kaggle/input/"
            base_output_path = "/kaggle/working/"
            environment_name = "kaggle"
        else:
            print("⚠️ Environment: Local/Unknown")
            base_data_path = "./data/"
            base_output_path = "./output/"
            environment_name = "local"
    except NameError:
        print("⚠️ Non-interactive session. Using local paths.")
        base_data_path = "./data/"
        base_output_path = "./output/"
        environment_name = "local"
    os.makedirs(base_output_path, exist_ok=True)
    print(f"📂 Data Path: {base_data_path}")
    print(f"📦 Output Path: {base_output_path}")
    return base_data_path, base_output_path, environment_name


def load_secret(key_name: str) -> str | None:
    env = ENV_NAME
    secret_value = None
    print(f"Attempting to load secret '{key_name}' from '{env}' environment...")
    try:
        if env == "colab":
            from google.colab import userdata

            secret_value = userdata.get(key_name)
        elif env == "kaggle":
            from kaggle_secrets import UserSecretsClient

            user_secrets = UserSecretsClient()
            secret_value = user_secrets.get_secret(key_name)
        else:
            secret_value = os.getenv(key_name)
        if not secret_value:
            print(f"⚠️ Secret '{key_name}' not found in the {env} environment.")
            return None
        print(f"✅ Successfully loaded secret '{key_name}'.")
        return secret_value
    except Exception as e:
        print(f"❌ An error occurred while loading secret '{key_name}': {e}")
        return None


def print_system_info():
    print("\n🔧 System Information")
    print(f"Python version: {sys.version.split()[0]}")
    try:
        import torch

        print(f"PyTorch version: {torch.__version__}")
        if torch.cuda.is_available():
            print(f"CUDA version: {torch.version.cuda}")
            print(f"GPU count: {torch.cuda.device_count()}")
            for i in range(torch.cuda.device_count()):
                print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        else:
            print("CUDA not available")
    except ImportError:
        print("PyTorch not installed")
    finally:
        !nvidia-smi


INPUT_PATH, OUTPUT_PATH, ENV_NAME = configure_environment_paths()
is_kaggle = "kaggle" in ENV_NAME
is_colab = not is_kaggle
print_system_info()

os.environ["WANDB_API_KEY"] = wandb_key = load_secret("WANDB_API_KEY")
os.environ["HF_TOKEN"] = HF_TOKEN = load_secret("HF_TOKEN")
GITHUB_TOKEN = load_secret("GITHUB_TOKEN")

✅ Environment: Google Colab
📂 Data Path: /content/
📦 Output Path: /content/

🔧 System Information
Python version: 3.12.13
PyTorch version: 2.11.0+cu130
CUDA version: 13.0
GPU count: 1
  GPU 0: Tesla T4
Thu Jun 18 13:18:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             

In [6]:
%cd {OUTPUT_PATH}
!rm -rf ToolFormer
!git clone https://{GITHUB_TOKEN}@github.com/dzungphieuluuky/ToolFormer.git
%cd {OUTPUT_PATH}/ToolFormer

/content
Cloning into 'ToolFormer'...
remote: Enumerating objects: 208, done.
remote: Counting objects: 100% (208/208), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 208 (delta 100), reused 177 (delta 75), pack-reused 0 (from 0)
Receiving objects: 100% (208/208), 1.40 MiB | 8.99 MiB/s, done.
Resolving deltas: 100% (100/100), done.
/content/ToolFormer


In [7]:
!python scripts/run_sft.py

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/trl/import_utils.py", line 144, in _get_module
    return importlib.import_module("." + module_name, self.__name__)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/usr/local/lib/python3.12/dist-packages/trl/trainer/grpo_trainer.py", line 54, in <m